# Signature verification - 256x512 training and evaluation (single run, no augmentation)

Trains the Stage 3 ConvNeXt-Tiny metric-learning encoder and evaluates it on a Kaggle T4.

**Canonical canvas:** `256 x 512` (H x W), grayscale, white background. This notebook
does not use the legacy 384x192 upstream.

**Dataset.** Attach the cleaned 256x512 dataset (`manifest_aug.csv` at the root, with
`BHSig260/`, `CEDAR/` and `SigResize/` beside it). Expected: 20,548 rows (14,401 train /
3,114 val / 3,033 test), 421 signers.

**Settings:** GPU **T4**, **Internet ON** (pretrained ConvNeXt weights), outputs under
`/kaggle/working`.

**Why one run.** The earlier A/B comparison on this split showed no benefit from
augmentation: test skilled EER 9.4% without it versus 12.8% with it, and the no-augmentation
run was better on BHSig260-Bengali, BHSig260-Hindi and CEDAR. This notebook therefore trains
only the no-augmentation model (`--no-augment`), which halves the training time.

Checkpoints are selected on validation skilled-forgery EER with **3 medoid-selected
references**, the same protocol evaluation reports. Evaluation sweeps **2 and 3 references**.
Fusion is off by default (`RUN_FUSION = False` in the fusion section).

In [ ]:
# Cell 1 - clone the repository and confirm the evaluation flags exist.
REPO_URL = "https://github.com/yogeshY0/signautre-256-512.git"
REPO_DIR = "/kaggle/working/repo"

!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1
!grep -q "reference-counts" training/evaluate.py && echo "code OK: reference sweep present"

!pip install -q --no-input timm opencv-python scikit-learn

In [ ]:
# Cell 2 - environment check.
import platform
import sys

import cv2
import timm
import torch

print("python :", sys.version.split()[0], platform.platform())
print("torch  :", torch.__version__, "| cuda:", torch.cuda.is_available())
print("gpu    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("timm   :", timm.__version__, "| cv2:", cv2.__version__)
assert torch.cuda.is_available(), "Select the T4 GPU accelerator before running"

In [ ]:
# Cell 3 - locate the attached dataset.
from pathlib import Path

PREFERRED_ROOT = Path("/kaggle/input/datasets/yogeshbaral/training-in-256512")

if (PREFERRED_ROOT / "manifest_aug.csv").is_file():
    IMAGE_ROOT = PREFERRED_ROOT
    MANIFEST = IMAGE_ROOT / "manifest_aug.csv"
else:
    candidates = sorted(Path("/kaggle/input").rglob("manifest_aug.csv"))
    assert candidates, "no manifest_aug.csv under /kaggle/input - attach the 256x512 dataset"
    MANIFEST = candidates[0]
    IMAGE_ROOT = MANIFEST.parent

assert MANIFEST.is_file(), f"manifest is not a file: {MANIFEST}"
print("manifest  :", MANIFEST)
print("image root:", IMAGE_ROOT)
print("top level :", sorted(path.name for path in IMAGE_ROOT.iterdir()))

In [ ]:
# Cell 4 - counts and path resolution. Fails before anything is trained.
import collections
import csv

rows = list(csv.DictReader(MANIFEST.open(encoding="utf-8")))
print("rows    :", len(rows))
print("splits  :", dict(collections.Counter(row["split"] for row in rows)))
print("signers :", len({row["signer_id"] for row in rows}))
print("datasets:", dict(collections.Counter(row["dataset"] for row in rows)))
missing = [
    row["image_path"] for row in rows if not (IMAGE_ROOT / row["image_path"]).is_file()
]
print("missing :", len(missing), missing[:3])
assert len(rows) == 20548, f"expected 20548 rows, got {len(rows)}"
assert not missing, "manifest paths do not resolve under IMAGE_ROOT"
print("PASS: manifest and image paths")

In [ ]:
# Cell 5 - data gate. Training must not start unless this prints PASS.
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "training.verify_data",
        "--manifest",
        str(MANIFEST),
        "--image-root",
        str(IMAGE_ROOT),
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
assert result.returncode == 0 and "RESULT: PASS" in result.stdout, (
    "verify_data did not pass; do not train on this dataset"
)
print("PASS: verify_data (3/3 sections)")

## 1. Train (no augmentation)

`--phase1-epochs 5 --phase2-epochs PHASE2_EPOCHS` = 5 warmup epochs with a frozen backbone and
semi-hard mining, then fine-tune epochs on the last two ConvNeXt stages with
batch-hard mining and cosine LR. `--select-by eer` keeps the checkpoint with the lowest
validation skilled-forgery EER. References are **3, medoid-selected**; the evaluation
sweeps 2 and 3. `--balance-datasets` stays off.

`--patience` ends the fine-tune phase early when the validation EER has not improved for that
many epochs. The best checkpoint is still the one saved at the lowest validation EER, so
stopping early never loses it. In the earlier 40-epoch run the best validation EER was at
fine-tune epoch 28 and the later epochs were flat, so patience 8 would typically stop around
epoch 36. Set `PATIENCE = 0` to always run all 40.

In [ ]:
# Cell 6 - shared settings.
SEED = 42
PHASE2_EPOCHS = 40   # full schedule; PATIENCE below can end it earlier
PATIENCE = 8         # stop fine-tuning after 8 epochs with no better validation EER (0 = off)
COMMON = (
    f"--manifest {MANIFEST} --image-root {IMAGE_ROOT} --already-preprocessed "
    f"--image_height 256 --image_width 512 --device cuda --workers 2 --seed {SEED} "
    f"--phase1-epochs 5 --phase2-epochs {PHASE2_EPOCHS} --patience {PATIENCE} "
    "--select-by eer --references 3 --reference-selection medoid"
)
print(COMMON)

In [ ]:
# Cell 6b - guard: stop before training if any signer is too small for the reference protocol.
# The old crash ("too few genuine signatures for 5 references") came from signers with
# fewer genuine images than references + 1. Training uses 3 references, so every
# validation and test signer needs at least 4 genuine images.
REFERENCES = 3
genuine_counts = collections.defaultdict(int)
signer_split = {}
for row in rows:
    signer_split[row["signer_id"]] = row["split"]
    if row["is_forgery"].strip().lower() not in ("true", "1", "yes"):
        genuine_counts[row["signer_id"]] += 1

too_small = sorted(
    (signer, genuine_counts[signer], signer_split[signer])
    for signer in signer_split
    if signer_split[signer] in ("val", "validation", "test")
    and genuine_counts[signer] < REFERENCES + 1
)
print("val/test signers with fewer than", REFERENCES + 1, "genuine images:", too_small)
assert not too_small, "some val/test signers cannot support 3 references; fix the data first"
print("PASS: every val/test signer has enough genuine images")

In [ ]:
# Cell 7 - train: no augmentation.
!python -u -m training.train_siamese {COMMON} --no-augment \
  --output-dir /kaggle/working/ckpt_A_noaug 2>&1 | tee /kaggle/working/train_A.log

In [ ]:
# Cell 7b - confirm the best weights were saved, and keep a safe copy right away.
import shutil
import torch

BEST = Path("/kaggle/working/ckpt_A_noaug/best_model.pt")
assert BEST.is_file(), "best_model.pt was not written - training did not save a checkpoint"
ckpt = torch.load(BEST, map_location="cpu")
print("saved at   :", ckpt["phase"], "epoch", ckpt["epoch"], "(global epoch count)")
print("val skilled EER:", round(float(ckpt["best_validation_eer_skilled"]), 4))
print("references :", ckpt["references"], "|", ckpt["reference_selection"])
print("augmentation:", ckpt["augmentation"])
assert ckpt["augmentation"]["enabled"] is False, "this run was meant to have augmentation off"
shutil.copy(BEST, "/kaggle/working/best_model_A_backup.pt")
print("backup copy: /kaggle/working/best_model_A_backup.pt")

## 2. Evaluate on the validation and test splits

These are the deep scores, with no fusion. The threshold is chosen on
validation signers and applied unchanged to test. The report contains AUC, EER,
FAR/FRR at `--target-far 0.001`, the **10% FRR operating point**, 95% signer-level
bootstrap CIs, per-dataset results (BHSig260-Hindi, BHSig260-Bengali, CEDAR, SigResize)
and the worst writers. References are swept over **2 and 3**; 3 is primary.

In [ ]:
# Cell 8 - evaluate on the validation and test splits.
import json
import subprocess

Path("/kaggle/working/reports").mkdir(parents=True, exist_ok=True)
CHECKPOINTS = {
    "A_noaug": "/kaggle/working/ckpt_A_noaug/best_model.pt",
}
REPORTS = {}
for name, checkpoint in CHECKPOINTS.items():
    report_path = f"/kaggle/working/reports/eval_{name}.json"
    print(f"\n=== evaluate {name} ===", flush=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "training.evaluate",
            "--checkpoint",
            checkpoint,
            "--manifest",
            str(MANIFEST),
            "--image-root",
            str(IMAGE_ROOT),
            "--already-preprocessed",
            "--image_height",
            "256",
            "--image_width",
            "512",
            "--references",
            "3",
            "--reference-counts",
            "2,3",
            "--reference-selection",
            "medoid",
            "--device",
            "cuda",
            "--batch-size",
            "32",
            "--workers",
            "2",
            "--target-far",
            "0.001",
            "--target-frr",
            "0.10",
            "--bootstrap",
            "1000",
            "--fusion-report",
            report_path,
        ],
        check=True,
    )
    REPORTS[name] = json.loads(Path(report_path).read_text())
print("reports:", sorted(REPORTS))

In [ ]:
# Cell 9 - results table and 2/3-reference sweep.
import pandas as pd

table = []
for name, report in REPORTS.items():
    row = {
        "run": name,
        "val_skilled": report["validation_eer_skilled"],
        "val_auc": report["validation_auc_skilled"],
        "test_skilled": report["test"]["eer_skilled"],
        "test_auc": report["test"].get("auc_skilled"),
        "test_random": report["test"]["eer_random"],
        "skilled@10%": report["operating_point_frr"]["deep"]["test"]["far_skilled_at_threshold"],
        "random@10%": report["operating_point_frr"]["deep"]["test"]["far_random_at_threshold"],
        "frr@10%": report["operating_point_frr"]["deep"]["test"]["frr_at_threshold"],
        "eer_ci": report["confidence_intervals"]["deep"]["pooled"]["eer_skilled"],
    }
    for dataset, metrics in report["per_dataset"]["test"].items():
        row[f"test_{dataset}"] = metrics["eer_skilled"]
    table.append(row)

comparison = pd.DataFrame(table).set_index("run").sort_index()
print(comparison.round(4).to_string())

sweep_rows = []
for name, report in REPORTS.items():
    for refs, block in report["reference_sweep"].items():
        sweep_rows.append(
            {
                "run": name,
                "references": int(refs),
                "val_EER": block["validation_eer_skilled"],
                "val_AUC": block["validation_auc_skilled"],
                "test_EER": block["test"]["eer_skilled"],
                "test_AUC": block["test"].get("auc_skilled"),
            }
        )
print("\nReference-count sweep (deep):")
print(pd.DataFrame(sweep_rows).sort_values(["run", "references"]).round(4).to_string(index=False))

## 3. Fusion (optional, off by default)

Fusion adds nine handcrafted features to the deep score. On the earlier model it moved the
validation EER from 10.09% to 9.67%, a small change, and it was never confirmed on test.
It is therefore skipped unless you set `RUN_FUSION = True`. It is fitted on this model's
validation split with **skilled-forgery negatives only** (`--max-random-queries 0`), using
the same 3 medoid references. Its `deep_vs_fused` verdict recommends `deep_only` unless
fusion improves **every** corpus. The evaluation step now handles the empty random-negative
set, so it no longer crashes.

In [ ]:
# Cell 10 - fit fusion on the validation split (only when RUN_FUSION is True).
RUN_FUSION = False

if RUN_FUSION:
    subprocess.run(
        [
            sys.executable, "-m", "training.fit_fusion",
            "--checkpoint", "/kaggle/working/ckpt_A_noaug/best_model.pt",
            "--manifest", str(MANIFEST), "--image-root", str(IMAGE_ROOT),
            "--already-preprocessed",
            "--image_height", "256", "--image_width", "512",
            "--references", "3", "--reference-selection", "medoid",
            "--device", "cuda", "--batch-size", "32", "--workers", "2",
            "--max-random-queries", "0",
            "--output", "/kaggle/working/fusion_A.json",
        ],
        check=True,
    )
else:
    print("fusion skipped (RUN_FUSION = False)")

In [ ]:
# Cell 11 - evaluate deep vs fused, with calibration and the fusion verdict.
if RUN_FUSION:
    fusion_report_path = "/kaggle/working/reports/eval_A_fused.json"
    subprocess.run(
        [
            sys.executable, "-m", "training.evaluate",
            "--checkpoint", "/kaggle/working/ckpt_A_noaug/best_model.pt",
            "--manifest", str(MANIFEST), "--image-root", str(IMAGE_ROOT),
            "--already-preprocessed",
            "--image_height", "256", "--image_width", "512",
            "--references", "3", "--reference-counts", "3",
            "--reference-selection", "medoid",
            "--device", "cuda", "--batch-size", "32", "--workers", "2",
            "--max-random-queries", "0",
            "--target-far", "0.001", "--target-frr", "0.10", "--bootstrap", "1000",
            "--fusion-config", "/kaggle/working/fusion_A.json",
            "--fusion-report", fusion_report_path,
        ],
        check=True,
    )

    fused_report = json.loads(Path(fusion_report_path).read_text())
    verdict = fused_report["deep_vs_fused"]
    print("FUSION VERDICT:", verdict["recommendation"],
          "| helps every dataset:", verdict["helps_every_dataset"])
    print("global vs per-dataset calibration (deep):")
    for dataset, entry in fused_report["calibration"]["deep"]["datasets"].items():
        global_rates = entry["test_at_global_threshold"]
        dataset_rates = entry["test_at_dataset_threshold"]
        print(
            f"  {dataset:<20} thr_ds={entry['threshold']:.4f} "
            f"FRR_global={global_rates['frr_at_threshold']:.4f} "
            f"FRR_ds={dataset_rates['frr_at_threshold']:.4f}"
        )
else:
    print("fused evaluation skipped (RUN_FUSION = False)")

## 4. Per-corpus audit

Same 3-reference medoid protocol as evaluation, with per-dataset EER, FAR/FRR at fixed
budgets and score histograms. Read-only.

In [ ]:
# Cell 12 - per-dataset audit.
!python -u -m training.audit_embeddings \
  --checkpoint /kaggle/working/ckpt_A_noaug/best_model.pt \
  --manifest {MANIFEST} --image-root {IMAGE_ROOT} --already-preprocessed \
  --image_height 256 --image_width 512 --references 3 --reference-selection medoid \
  --device cuda --batch-size 32 --workers 2 \
  --max-random-queries 2000 --target-fars 0.001,0.01 \
  --hist-dir /kaggle/working/audit_histograms

## 5. Package the artifacts

`/kaggle/working` is what Kaggle offers for download when the session ends.

In [ ]:
# Cell 13 - tar the checkpoint, reports and log (and fusion if it was fitted).
import os

files = ["ckpt_A_noaug/best_model.pt", "train_A.log", "reports"]
if os.path.exists("/kaggle/working/fusion_A.json"):
    files.append("fusion_A.json")
subprocess.run(
    ["tar", "-czf", "/kaggle/working/artifacts_256x512.tar.gz",
     "-C", "/kaggle/working", *files],
    check=True,
)
!ls -lh /kaggle/working/artifacts_256x512.tar.gz /kaggle/working/ckpt_*/best_model.pt

## Reading the results

* Report **`per_dataset.test.<corpus>.eer_skilled`**, not only the pooled average. The
  groups are `BHSig260-Hindi`, `BHSig260-Bengali`, `CEDAR` and `SigResize`.
* `test.auc_skilled` and `confidence_intervals.deep.pooled` are the AUC and 95% CI.
* `operating_point_frr.deep.test` is the 10% FRR operating point; compare skilled and
  random acceptance, not only EER.
* `per_signer.deep.worst_frr` / `worst_far` list the weakest writers.
* `reference_sweep` covers 2 and 3 references; 3 is primary. The earlier run gave
  similar test EER for 2 and 3 references (8.7% vs 9.4%) and the gap is inside the CI.
* `deep_vs_fused.recommendation` must be `keep_fusion` before a fused scorer ships
  (only produced when `RUN_FUSION = True`).
* Do not quote metrics from a 1-epoch, random-init smoke run.
* Download `artifacts_256x512.tar.gz` before the session ends.

**Canonical canvas:** 256 x 512. Do not merge the old 384 x 192 upstream into this branch.